In [6]:
# import pandas as pd
# import numpy as np
# import xgboost as xgb
# from sklearn.metrics import (
#     mean_absolute_error, 
#     mean_squared_error, 
#     r2_score,
#     mean_absolute_percentage_error,
#     median_absolute_error,
#     max_error,
#     explained_variance_score
# )

# # 1. Load the training data (assuming you saved it as train.ods)
# print("Loading training data...")
# df_train = pd.read_excel('train.ods', engine='odf')
# df_train['Calendar'] = pd.to_datetime(df_train['Calendar'])

# # CRITICAL: Sort by date before doing anything else
# df_train = df_train.sort_values('Calendar').reset_index(drop=True)

# # 2. Feature Engineering: Extract numeric features from the Date
# def extract_date_features(df):
#     df = df.copy()
#     df['Year'] = df['Calendar'].dt.year
#     df['Month'] = df['Calendar'].dt.month
#     df['Day'] = df['Calendar'].dt.day
#     df['DayOfWeek'] = df['Calendar'].dt.dayofweek
#     df['DayOfYear'] = df['Calendar'].dt.dayofyear
#     return df

# df_train = extract_date_features(df_train)

# # 3. Define Features and Target
# # We drop 'Calendar' because XGBoost only takes numbers
# features = ['Crude_Price_USD', 'WPI_Paddy', 'Yield', 'MSP_Fixed', 
#             'Year', 'Month', 'Day', 'DayOfWeek', 'DayOfYear']
# target = 'Mandi Modal Price (AgMarknet)'

# X = df_train[features]
# y = df_train[target]

# # 4. Chronological Train/Validation Split (80% Train, 20% Validate)
# split_index = int(len(df_train) * 0.8)

# X_train, X_val = X.iloc[:split_index], X.iloc[split_index:]
# y_train, y_val = y.iloc[:split_index], y.iloc[split_index:]

# # 5. Initialize and Train the XGBoost Regressor
# print("Training XGBoost Model...")
# model = xgb.XGBRegressor(
#     n_estimators=1000,       # Number of trees
#     learning_rate=0.05,      # Step size
#     max_depth=6,             # Complexity of each tree
#     random_state=42,
#     early_stopping_rounds=50 # Stops training if validation score doesn't improve
# )

# # Fit the model, evaluating on the validation set to prevent overfitting
# model.fit(
#     X_train, y_train,
#     eval_set=[(X_train, y_train), (X_val, y_val)],
#     verbose=False
# )

# # 6. Get Metrics
# predictions = model.predict(X_val)

# mae = mean_absolute_error(y_val, predictions)
# rmse = np.sqrt(mean_squared_error(y_val, predictions))
# mape = mean_absolute_percentage_error(y_val, predictions) * 100  # Multiplied by 100 for standard % format
# medae = median_absolute_error(y_val, predictions)
# max_err = max_error(y_val, predictions)
# r2 = r2_score(y_val, predictions)
# evs = explained_variance_score(y_val, predictions)

# print("\n--- EXTENSIVE MODEL METRICS (Validation Set) ---")
# print(f"MAE (Mean Absolute Error):        ₹{mae:.2f}")
# print(f"MedAE (Median Absolute Error):    ₹{medae:.2f}")
# print(f"Max Error (Worst Prediction):     ₹{max_err:.2f}")
# print('-' * 48)
# print(f"RMSE (Root Mean Squared Error):   ₹{rmse:.2f}")
# print(f"MAPE (Mean Abs Percentage Error): {mape:.2f}%")
# print('-' * 48)
# print(f"R-squared:                        {r2:.4f}")
# print(f"Explained Variance Score:         {evs:.4f}")


import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error, median_absolute_error,
    max_error, explained_variance_score
)

# ── 1. Load & Sort ────────────────────────────────────────────────────────────
print("Loading training data...")
df = pd.read_excel('train.ods', engine='odf')
df['Calendar'] = pd.to_datetime(df['Calendar'])
df = df.sort_values('Calendar').reset_index(drop=True)

TARGET = 'Mandi Modal Price (AgMarknet)'

# ── 2. Lag Features (THE most important addition) ─────────────────────────────
# These give the model "memory" of where prices have been
lag_days = [1, 2, 3, 7, 14, 21, 30]
for lag in lag_days:
    df[f'price_lag_{lag}'] = df[TARGET].shift(lag)

# ── 3. Rolling Window Statistics ──────────────────────────────────────────────
# Capture short-term and medium-term trend
for window in [7, 14, 30]:
    df[f'rolling_mean_{window}'] = df[TARGET].shift(1).rolling(window).mean()
    df[f'rolling_std_{window}']  = df[TARGET].shift(1).rolling(window).std()
    df[f'rolling_min_{window}']  = df[TARGET].shift(1).rolling(window).min()
    df[f'rolling_max_{window}']  = df[TARGET].shift(1).rolling(window).max()

# ── 4. Price Momentum / Trend ─────────────────────────────────────────────────
df['price_change_1d']  = df[TARGET].shift(1) - df[TARGET].shift(2)
df['price_change_7d']  = df[TARGET].shift(1) - df[TARGET].shift(7)
df['price_change_30d'] = df[TARGET].shift(1) - df[TARGET].shift(30)

# ── 5. Calendar Features ──────────────────────────────────────────────────────
df['Year']      = df['Calendar'].dt.year
df['Month']     = df['Calendar'].dt.month
df['Day']       = df['Calendar'].dt.day
df['DayOfWeek'] = df['Calendar'].dt.dayofweek
df['DayOfYear'] = df['Calendar'].dt.dayofyear
df['Quarter']   = df['Calendar'].dt.quarter
df['WeekOfYear']= df['Calendar'].dt.isocalendar().week.astype(int)

# ── 6. Drop rows where lags are NaN (first ~30 rows) ─────────────────────────
df = df.dropna().reset_index(drop=True)
print(f"Rows after dropping NaN lag rows: {len(df)}")

# ── 7. Define Features ────────────────────────────────────────────────────────
lag_features     = [f'price_lag_{l}' for l in lag_days]
rolling_features = [f'rolling_{stat}_{w}' 
                    for stat in ['mean','std','min','max'] 
                    for w in [7,14,30]]
momentum_features = ['price_change_1d', 'price_change_7d', 'price_change_30d']
calendar_features = ['Year','Month','Day','DayOfWeek','DayOfYear','Quarter','WeekOfYear']
external_features = ['Crude_Price_USD', 'WPI_Paddy', 'Yield', 'MSP_Fixed']

features = lag_features + rolling_features + momentum_features + \
           calendar_features + external_features

X = df[features]
y = df[TARGET]

# ── 8. Chronological Split (80/20) ────────────────────────────────────────────
split_idx = int(len(df) * 0.8)
X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]
print(f"Train: {len(X_train)} rows | Val: {len(X_val)} rows")

# ── 9. Train XGBoost ──────────────────────────────────────────────────────────
print("\nTraining XGBoost Model...")
model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,       # Slower learning = better generalisation
    max_depth=5,              # Slightly shallower to reduce overfit
    min_child_weight=5,       # Prevents learning from tiny samples
    subsample=0.8,            # Row sampling per tree (regularisation)
    colsample_bytree=0.8,     # Feature sampling per tree (regularisation)
    gamma=0.1,                # Min loss reduction to make a split
    reg_alpha=0.1,            # L1 regularisation
    reg_lambda=1.0,           # L2 regularisation
    random_state=42,
    early_stopping_rounds=75
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=200
)

# ── 10. Metrics ───────────────────────────────────────────────────────────────
predictions = model.predict(X_val)

mae   = mean_absolute_error(y_val, predictions)
rmse  = np.sqrt(mean_squared_error(y_val, predictions))
mape  = mean_absolute_percentage_error(y_val, predictions) * 100
medae = median_absolute_error(y_val, predictions)
max_e = max_error(y_val, predictions)
r2    = r2_score(y_val, predictions)
evs   = explained_variance_score(y_val, predictions)

print("\n--- MODEL METRICS (Validation Set) ---")
print(f"MAE (Mean Absolute Error):        ₹{mae:.2f}")
print(f"MedAE (Median Absolute Error):    ₹{medae:.2f}")
print(f"Max Error (Worst Prediction):     ₹{max_e:.2f}")
print(f"RMSE:                             ₹{rmse:.2f}")
print(f"MAPE:                             {mape:.2f}%")
print(f"R-squared:                        {r2:.4f}")
print(f"Explained Variance Score:         {evs:.4f}")

# ── 11. Feature Importance (top 15) ──────────────────────────────────────────
feat_imp = pd.Series(model.feature_importances_, index=features)
print("\n--- TOP 15 FEATURES ---")
print(feat_imp.sort_values(ascending=False).head(15).to_string())

Loading training data...
Rows after dropping NaN lag rows: 4163
Train: 3330 rows | Val: 833 rows

Training XGBoost Model...
[0]	validation_0-rmse:236.52610	validation_1-rmse:388.46647
[200]	validation_0-rmse:59.22385	validation_1-rmse:58.08063
[362]	validation_0-rmse:48.16671	validation_1-rmse:58.90665

--- MODEL METRICS (Validation Set) ---
MAE (Mean Absolute Error):        ₹38.76
MedAE (Median Absolute Error):    ₹27.75
Max Error (Worst Prediction):     ₹384.07
RMSE:                             ₹57.89
MAPE:                             2.08%
R-squared:                        0.9111
Explained Variance Score:         0.9116

--- TOP 15 FEATURES ---
rolling_mean_7     0.485068
rolling_mean_14    0.277814
rolling_min_7      0.041225
price_lag_1        0.026470
price_lag_7        0.026308
rolling_min_14     0.008532
rolling_mean_30    0.007795
price_lag_14       0.007210
rolling_max_7      0.006130
price_lag_2        0.006033
Month              0.005851
Yield              0.005749
price_la

In [8]:
# def generate_submission(test_filepath, trained_model, historical_df):
#     """
#     Interface to predict prices for a given test file.
#     """
#     print(f"\nProcessing judge's test file: {test_filepath}")
    
#     # 1. Load the test file
#     df_test = pd.read_excel(test_filepath, engine='odf')
#     df_test['Calendar'] = pd.to_datetime(df_test['Calendar'])
    
#     # 2. Add Date Features
#     df_test = extract_date_features(df_test)
    
#     # 3. Handle Macro Features (The Tricky Part!)
#     # If the test file is in the future, we don't know the future Crude or WPI.
#     # Standard practice is to "forward fill" the last known macro values from your training data.
    
#     # Get the most recent macro values from the training set
#     last_known_values = historical_df.iloc[-1]
    
#     # If the judge's file doesn't have these columns, we populate them with the last known data
#     for col in ['Crude_Price_USD', 'WPI_Paddy', 'Yield', 'MSP_Fixed']:
#         if col not in df_test.columns:
#             df_test[col] = last_known_values[col]
            
#     # 4. Ensure column order matches the training data exactly
#     X_test = df_test[features]
    
#     # 5. Predict
#     df_test['Predicted_Price'] = trained_model.predict(X_test)
    
#     # 6. Save the output
#     submission = df_test[['Calendar', 'Predicted_Price']]
#     submission.to_csv('submission.csv', index=False)
    
#     print("Predictions complete! Saved to 'submission.csv'")
#     return submission

# # --- Run the Interface ---
# # (Uncomment this when the judge provides the file)
# final_predictions = generate_submission('judge_test.ods', model, df_train)

def generate_submission(test_filepath, trained_model, historical_df, feature_cols):
    """
    Generates predictions for a future test file.
    Handles lag/rolling features by iteratively building price history.
    
    Args:
        test_filepath  : Path to judge's .ods test file
        trained_model  : The fitted XGBRegressor
        historical_df  : Your full training DataFrame (with TARGET column populated)
        feature_cols   : The `features` list from training (ensures column order matches)
    """
    TARGET = 'Mandi Modal Price (AgMarknet)'

    print(f"\nProcessing test file: {test_filepath}")

    # ── 1. Load test file ─────────────────────────────────────────────────────
    df_test = pd.read_excel(test_filepath, engine='odf')
    df_test['Calendar'] = pd.to_datetime(df_test['Calendar'])
    df_test = df_test.sort_values('Calendar').reset_index(drop=True)
    print(f"  Test rows to predict: {len(df_test)}")

    # ── 2. Forward-fill missing macro features from last known training values ─
    last_known = historical_df.iloc[-1]
    for col in ['Crude_Price_USD', 'WPI_Paddy', 'Yield', 'MSP_Fixed']:
        if col not in df_test.columns or df_test[col].isna().all():
            df_test[col] = last_known[col]
            print(f"  Forward-filled '{col}' with last known value: {last_known[col]:.2f}")

    # ── 3. Build a running history buffer (real prices + predicted prices) ────
    # Start with the last 60 rows of training data — enough for all lag/rolling windows
    BUFFER_SIZE = 60
    history_buffer = historical_df[[TARGET, 'Calendar']].tail(BUFFER_SIZE).copy()
    history_buffer = history_buffer.reset_index(drop=True)

    # ── 4. Iterative Prediction ───────────────────────────────────────────────
    # For each test row, compute features from the running buffer, predict,
    # then append the prediction to the buffer for the next row.
    predicted_prices = []
    lag_days         = [1, 2, 3, 7, 14, 21, 30]
    rolling_windows  = [7, 14, 30]

    print("\n  Predicting row by row...")
    for i, row in df_test.iterrows():
        price_series = history_buffer[TARGET]

        # ── Lag features ──────────────────────────────────────────────────────
        row_features = {}
        for lag in lag_days:
            idx = len(price_series) - lag
            row_features[f'price_lag_{lag}'] = price_series.iloc[idx] if idx >= 0 else np.nan

        # ── Rolling statistics (computed on available history) ────────────────
        for w in rolling_windows:
            window_data = price_series.iloc[-w:] if len(price_series) >= w else price_series
            row_features[f'rolling_mean_{w}'] = window_data.mean()
            row_features[f'rolling_std_{w}']  = window_data.std()
            row_features[f'rolling_min_{w}']  = window_data.min()
            row_features[f'rolling_max_{w}']  = window_data.max()

        # ── Momentum features ─────────────────────────────────────────────────
        p1  = price_series.iloc[-1]  if len(price_series) >= 1  else np.nan
        p2  = price_series.iloc[-2]  if len(price_series) >= 2  else np.nan
        p7  = price_series.iloc[-7]  if len(price_series) >= 7  else np.nan
        p30 = price_series.iloc[-30] if len(price_series) >= 30 else np.nan
        row_features['price_change_1d']  = p1 - p2
        row_features['price_change_7d']  = p1 - p7
        row_features['price_change_30d'] = p1 - p30

        # ── Calendar features ─────────────────────────────────────────────────
        dt = row['Calendar']
        row_features['Year']       = dt.year
        row_features['Month']      = dt.month
        row_features['Day']        = dt.day
        row_features['DayOfWeek']  = dt.dayofweek
        row_features['DayOfYear']  = dt.dayofyear
        row_features['Quarter']    = dt.quarter
        row_features['WeekOfYear'] = dt.isocalendar()[1]

        # ── External/macro features ───────────────────────────────────────────
        for col in ['Crude_Price_USD', 'WPI_Paddy', 'Yield', 'MSP_Fixed']:
            row_features[col] = row[col]

        # ── Predict ───────────────────────────────────────────────────────────
        X_row = pd.DataFrame([row_features])[feature_cols]  # enforce column order
        pred  = trained_model.predict(X_row)[0]
        predicted_prices.append(pred)

        # ── Append prediction to buffer so next row can use it as lag ─────────
        new_row = pd.DataFrame([{TARGET: pred, 'Calendar': dt}])
        history_buffer = pd.concat([history_buffer, new_row], ignore_index=True)

    # ── 5. Build & save submission ────────────────────────────────────────────
    submission = pd.DataFrame({
        'Calendar'        : df_test['Calendar'],
        'Predicted_Price' : predicted_prices
    })

    output_path = 'submission.csv'
    submission.to_csv(output_path, index=False)

    print(f"\n  ✓ {len(submission)} predictions saved to '{output_path}'")
    print(f"  Price range: ₹{min(predicted_prices):.2f} – ₹{max(predicted_prices):.2f}")
    print(f"  Mean predicted price: ₹{np.mean(predicted_prices):.2f}")
    return submission


# ── Run ───────────────────────────────────────────────────────────────────────
# Pass `features` (the list from training) to guarantee column order matches
final_predictions = generate_submission('judge_test.ods', model, df, features)


Processing test file: judge_test.ods
  Test rows to predict: 424
  Forward-filled 'Crude_Price_USD' with last known value: 95.31
  Forward-filled 'WPI_Paddy' with last known value: 187.00
  Forward-filled 'Yield' with last known value: 2.68
  Forward-filled 'MSP_Fixed' with last known value: 2040.00

  Predicting row by row...

  ✓ 424 predictions saved to 'submission.csv'
  Price range: ₹1970.10 – ₹2115.24
  Mean predicted price: ₹2064.99


In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_absolute_error, 
    mean_squared_error, 
    r2_score,
    mean_absolute_percentage_error,
    median_absolute_error,
    max_error,
    explained_variance_score
)

print("Loading test results from submission.csv...")

# 1. Load the updated submission file
df_results = pd.read_csv('submission.csv')

# Saftey check: Drop any rows where the Actual_Price might be accidentally blank
df_results = df_results.dropna(subset=['Actual_Price', 'Predicted_Price'])

# 2. Extract the actual and predicted columns
y_actual = df_results['Actual_Price']
y_pred = df_results['Predicted_Price']

# 3. Calculate all the metrics
mae = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
mape = mean_absolute_percentage_error(y_actual, y_pred) * 100  # Convert to percentage
medae = median_absolute_error(y_actual, y_pred)
max_err = max_error(y_actual, y_pred)
r2 = r2_score(y_actual, y_pred)
evs = explained_variance_score(y_actual, y_pred)

# 4. Print the final scorecard
print("\n--- FINAL TEST SET METRICS (submission.csv) ---")
print(f"MAE (Mean Absolute Error):        ₹{mae:.2f}")
print(f"MedAE (Median Absolute Error):    ₹{medae:.2f}")
print(f"Max Error (Worst Prediction):     ₹{max_err:.2f}")
print('-' * 48)
print(f"RMSE (Root Mean Squared Error):   ₹{rmse:.2f}")
print(f"MAPE (Mean Abs Percentage Error): {mape:.2f}%")
print('-' * 48)
print(f"R-squared:                        {r2:.4f}")
print(f"Explained Variance Score:         {evs:.4f}")

Loading test results from submission.csv...

--- FINAL TEST SET METRICS (submission.csv) ---
MAE (Mean Absolute Error):        ₹147.66
MedAE (Median Absolute Error):    ₹132.90
Max Error (Worst Prediction):     ₹612.22
------------------------------------------------
RMSE (Root Mean Squared Error):   ₹173.31
MAPE (Mean Abs Percentage Error): 6.62%
------------------------------------------------
R-squared:                        -1.3711
Explained Variance Score:         -0.2204
